# SAP Cloud ALM CDM Features API – References and Related Entities

This notebook covers the related-entity operations of the SAP Cloud ALM **CDM Features API** (`calm-features/v1`):

- **URL References** — link external web resources to a Feature (create, list, delete)
- **External References** — link tracker items (Jira, GitHub, etc.) using a stable composite key (create with idempotency, list, delete)
- **Task Assignments** — read-only list of project tasks, user stories, defects, and requirements linked to a Feature
- **Transports and Transport References** — read-only lists surfaced by the Features API

## Prerequisites

**Environment setup (one-time):**
1. Create a virtual environment: `python -m venv .venv`
2. Activate it: `source .venv/bin/activate` (macOS/Linux) or `.venv\Scripts\activate` (Windows)
3. Install dependencies: `pip install -r requirements.txt`

## Sandbox vs. Production

Set `USE_SANDBOX = True` in the setup cell to run against the public **SAP API Business Hub sandbox** — no tenant credentials required, GET cells only.

Set `USE_SANDBOX = False` to use your own SAP Cloud ALM tenant with OAuth2. Create an `apidata.py` file from `apidata_template.py`:

```python
# apidata.py
cdm_features_client_id     = 'your client ID'
cdm_features_client_secret = 'your client secret'
token_url = 'https://<identityzone>.authentication.<region>.hana.ondemand.com/oauth/token'
base_url  = 'https://<tenant>.<region>.alm.cloud.sap'
```

**Never commit `apidata.py`** — it contains credentials.

The **POST and DELETE cells** require production credentials — the sandbox does not support write operations.

In [ ]:
!python -m pip install -q requests
print("Dependencies ready")

## 1. Setup and Authentication

Set `USE_SANDBOX`, `SANDBOX_APIKEY`, and `FEATURE_UUID` before running.

When `USE_SANDBOX = True` and `FEATURE_UUID` is left as the placeholder, the cell auto-discovers a Feature UUID from the sandbox.

In [ ]:
import requests
from urllib.parse import urljoin
from uuid import UUID
import json

FEATURES_API_PATH = "/api/calm-features/v1/"
SANDBOX_BASE      = "https://sandbox.api.sap.com/SAPCALM/calm-features/v1/"

# ── Toggle ────────────────────────────────────────────────────────────────────
USE_SANDBOX    = True   # True = sandbox (read-only GET only); False = production (OAuth2)
SANDBOX_APIKEY = ""     # Required when USE_SANDBOX = True — get a free key at https://api.sap.com

# ── User-configurable IDs ─────────────────────────────────────────────────────
# Set to a valid Feature UUID from your tenant.
# In sandbox mode, leave the placeholder to auto-pick one from the sandbox.
FEATURE_UUID = "<your-feature-uuid>"

# External Reference stable id used in Sections 4 and 5.
EXT_REF_ID = "jira:ABC-123"  # change to match your tracker reference

# ── Helpers ───────────────────────────────────────────────────────────────────
def normalize_tenant_base(raw_base_url: str) -> str:
    base = raw_base_url.strip().rstrip("/")
    lowered = base.lower()
    for marker in ("/api/calm-features/v1/features", "/api/calm-features/v1"):
        pos = lowered.find(marker)
        if pos != -1:
            return base[:pos].rstrip("/")
    return base

def get_token(token_url, client_id, client_secret):
    r = requests.post(
        token_url,
        data={"grant_type": "client_credentials"},
        auth=(client_id, client_secret),
        timeout=30,
    )
    r.raise_for_status()
    return r.json()["access_token"]

# ── Auth ──────────────────────────────────────────────────────────────────────
if USE_SANDBOX:
    if not SANDBOX_APIKEY.strip():
        raise ValueError("Set SANDBOX_APIKEY to your API key from https://api.sap.com")
    BASE_URL     = SANDBOX_BASE
    HEADERS      = {"apikey": SANDBOX_APIKEY}
    JSON_HEADERS = {**HEADERS, "Content-Type": "application/json"}
    print("Mode: SANDBOX (read-only GET cells only)")
else:
    import apidata as ad
    token        = get_token(ad.token_url, ad.cdm_features_client_id, ad.cdm_features_client_secret)
    tenant_base  = normalize_tenant_base(ad.base_url)
    BASE_URL     = tenant_base + FEATURES_API_PATH
    HEADERS      = {"Authorization": f"Bearer {token}"}
    JSON_HEADERS = {**HEADERS, "Content-Type": "application/json"}
    print("Mode: PRODUCTION (token valid for 12 h — re-run this cell if you get HTTP 401)")

# ── FEATURE_UUID ──────────────────────────────────────────────────────────────
if FEATURE_UUID.strip() == "<your-feature-uuid>":
    if USE_SANDBOX:
        probe = requests.get(
            urljoin(BASE_URL, "Features"),
            headers=HEADERS,
            params={"$top": "1", "$select": "uuid,title"},
            timeout=30,
        )
        probe.raise_for_status()
        sample = probe.json().get("value", [])
        if not sample:
            raise ValueError("Sandbox returned no Features. Set FEATURE_UUID manually.")
        FEATURE_UUID = sample[0]["uuid"]
        print(f"Auto-selected sandbox FEATURE_UUID: {FEATURE_UUID} ('{sample[0].get('title', '')}')")
    else:
        raise ValueError(
            "FEATURE_UUID is empty. Paste a Feature UUID from your tenant. "
            "Run calm-cdm-features-api-v1.ipynb Section 4 to list Features and copy a UUID."
        )

FEATURE_UUID = str(UUID(FEATURE_UUID.strip()))
print(f"Base URL     : {BASE_URL}")
print(f"Feature UUID : {FEATURE_UUID}")
print(f"EXT_REF_ID   : {EXT_REF_ID}")

## 2. Feature Reference Overview (`$expand`)

The most efficient way to see **all** reference data for a Feature in a single API call is to use `$expand`:

```
GET /Features/{uuid}?$expand=toURLReferences,toExternalReferences
```

This avoids three separate round-trips and returns the full inline payload in one response. Run this cell as a quick overview before working with individual reference sections below.

In [ ]:
r = requests.get(
    urljoin(BASE_URL, f"Features/{FEATURE_UUID}"),
    headers=HEADERS,
    params={"$expand": "toURLReferences,toExternalReferences"},
    timeout=30,
)
if not r.ok:
    print(f"Request failed: HTTP {r.status_code}")
    print(r.text)
r.raise_for_status()
feature  = r.json()
url_refs = feature.get("toURLReferences", [])
ext_refs = feature.get("toExternalReferences", [])
print(f"URL References      : {len(url_refs)}")
print(f"External References : {len(ext_refs)}")
print(json.dumps(feature, indent=2))

## 3. URL References

URL References link any web URL to a Feature — useful for specification documents, design mockups, or runbooks.

- Each URL Reference has a server-generated `uuid`.
- To avoid duplicates, check existing references before creating.
- Endpoint pattern: `/Features/{uuid}/toURLReferences`

> **Sandbox:** POST and DELETE are not supported. Those cells are skipped when `USE_SANDBOX = True`.

In [ ]:
r = requests.get(
    urljoin(BASE_URL, f"Features/{FEATURE_UUID}/toURLReferences"),
    headers=HEADERS,
    timeout=30,
)
r.raise_for_status()
url_refs = r.json().get("value", [])
print(f"Existing URL References: {len(url_refs)}")
print(json.dumps(url_refs, indent=2))

In [ ]:
if USE_SANDBOX:
    print(
        "Sandbox mode is read-only: POST /Features/{uuid}/toURLReferences is not allowed. "
        "Set USE_SANDBOX = False and configure OAuth credentials in apidata.py to create URL references."
    )
else:
    new_url_ref = {
        "name": "Design Document",
        "url": "https://example.com/design",
    }
    r = requests.post(
        urljoin(BASE_URL, f"Features/{FEATURE_UUID}/toURLReferences"),
        headers=JSON_HEADERS,
        json=new_url_ref,
        timeout=30,
    )
    r.raise_for_status()
    created_url_ref = r.json()
    URL_REF_UUID = created_url_ref.get("uuid")
    print(f"Created URL Reference UUID: {URL_REF_UUID}")
    print(json.dumps(created_url_ref, indent=2))

In [ ]:
# GET a single URL Reference by UUID.
# In sandbox mode, auto-picks the first available reference (makes this cell standalone).
url_ref_uuid = globals().get("URL_REF_UUID")

if USE_SANDBOX and not url_ref_uuid:
    sample = requests.get(
        urljoin(BASE_URL, f"Features/{FEATURE_UUID}/toURLReferences"),
        headers=HEADERS,
        params={"$top": "1"},
        timeout=30,
    )
    sample.raise_for_status()
    sample_refs = sample.json().get("value", [])
    if sample_refs:
        url_ref_uuid = sample_refs[0].get("uuid")
        print(f"Using sandbox sample URL Reference UUID: {url_ref_uuid}")
    else:
        print("No URL References found for this Feature in sandbox — nothing to fetch.")

if not USE_SANDBOX and not url_ref_uuid:
    raise ValueError("URL_REF_UUID is not set. Run the create cell first or set it manually.")

if url_ref_uuid:
    r = requests.get(urljoin(BASE_URL, f"URLReferences/{url_ref_uuid}"), headers=HEADERS, timeout=30)
    r.raise_for_status()
    print(json.dumps(r.json(), indent=2))

In [ ]:
# Delete a URL Reference by its uuid.
# URL_REF_UUID is set by cell-urlref-create; set it manually if running this cell independently:
# URL_REF_UUID = "<paste-uuid-here>"

if USE_SANDBOX:
    print(
        "Sandbox mode is read-only: DELETE /URLReferences/{uuid} is not allowed. "
        "Set USE_SANDBOX = False and configure OAuth credentials in apidata.py to delete URL references."
    )
else:
    r = requests.delete(urljoin(BASE_URL, f"URLReferences/{URL_REF_UUID}"), headers=HEADERS, timeout=30)
    r.raise_for_status()
    print(f"Deleted URL Reference {URL_REF_UUID} (HTTP {r.status_code})")

## 4. External References

External References link issues from external trackers (Jira, GitHub, Azure DevOps, etc.) to a Feature.

### Key design: composite key `(id, parent_uuid)`

Unlike URL References, External References are identified by a **stable `id` you provide**, not a server-generated UUID.
The `id` must be unique per Feature and should encode the tracker system, e.g.:

| Tracker | Recommended `id` format | Example |
|---------|------------------------|----------|
| Jira | `jira:<PROJECT-KEY>` | `jira:ABC-123` |
| GitHub | `github:<repo>#<number>` | `github:org/repo#42` |
| Azure DevOps | `ado:<id>` | `ado:9876` |

The composite key `(id, parent_uuid)` means the same `id` value can be reused across different Features without conflict.

> **Sandbox:** POST and DELETE are not supported. Those cells are skipped when `USE_SANDBOX = True`.

In [ ]:
r = requests.get(
    urljoin(BASE_URL, f"Features/{FEATURE_UUID}/toExternalReferences"),
    headers=HEADERS,
    timeout=30,
)
r.raise_for_status()
ext_refs = r.json().get("value", [])
print(f"Existing External References: {len(ext_refs)}")
print(json.dumps(ext_refs, indent=2))

In [ ]:
# Idempotent create: check before POST to avoid duplicates.
# In sandbox mode, resolves EXT_REF_PARENT_UUID from existing references for downstream GET cells.
from urllib.parse import urlencode, quote

base_extref_url = urljoin(BASE_URL, f"Features/{FEATURE_UUID}/toExternalReferences")
filter_expr     = f"id eq '{EXT_REF_ID}'"
check_url       = base_extref_url + "?" + urlencode({"$filter": filter_expr}, quote_via=quote)
check           = requests.get(check_url, headers=HEADERS, timeout=30)

if check.status_code == 400:
    # Backend-specific fallback: fetch all and match client-side when $filter is rejected.
    print("Filtered lookup returned HTTP 400; falling back to full list + local match")
    list_resp = requests.get(base_extref_url, headers=HEADERS, timeout=30)
    list_resp.raise_for_status()
    existing = [x for x in list_resp.json().get("value", []) if x.get("id") == EXT_REF_ID]
else:
    check.raise_for_status()
    existing = check.json().get("value", [])

if USE_SANDBOX:
    if existing:
        EXT_REF_PARENT_UUID = existing[0].get("parent_uuid") or FEATURE_UUID
        print(f"Found existing External Reference '{EXT_REF_ID}' in sandbox:")
        print(json.dumps(existing[0], indent=2))
    else:
        # Fall back to first available sandbox reference so downstream GET cells can run.
        sample = requests.get(base_extref_url, headers=HEADERS, params={"$top": "1"}, timeout=30)
        sample.raise_for_status()
        sample_refs = sample.json().get("value", [])
        if sample_refs:
            EXT_REF_ID          = sample_refs[0].get("id", EXT_REF_ID)
            EXT_REF_PARENT_UUID = sample_refs[0].get("parent_uuid") or FEATURE_UUID
            print("Requested EXT_REF_ID not found. Reusing first existing sandbox reference:")
            print(json.dumps(sample_refs[0], indent=2))
        else:
            EXT_REF_PARENT_UUID = FEATURE_UUID
            print("No External References exist yet for this Feature in sandbox.")
    print(
        "\nSandbox mode is read-only: POST /toExternalReferences is not allowed. "
        "Set USE_SANDBOX = False and configure OAuth credentials in apidata.py to create external references."
    )
else:
    if existing:
        print(f"External Reference '{EXT_REF_ID}' already exists — skipping create")
        print(json.dumps(existing[0], indent=2))
        EXT_REF_PARENT_UUID = existing[0].get("parent_uuid")
    else:
        payload = {
            "id": EXT_REF_ID,
            "name": "Jira Story ABC-123",
            "url": "https://jira.example.com/browse/ABC-123",
        }
        r = requests.post(
            urljoin(BASE_URL, f"Features/{FEATURE_UUID}/toExternalReferences"),
            headers=JSON_HEADERS,
            json=payload,
            timeout=30,
        )
        r.raise_for_status()
        created_ext_ref = r.json()
        EXT_REF_PARENT_UUID = created_ext_ref.get("parent_uuid")
        print("Created External Reference:")
        print(json.dumps(created_ext_ref, indent=2))

print(f"\nEXT_REF_PARENT_UUID : {EXT_REF_PARENT_UUID}")
print(f"FEATURE_UUID        : {FEATURE_UUID}")

In [ ]:
# GET a single External Reference by composite key (id, parent_uuid).
# EXT_REF_ID and EXT_REF_PARENT_UUID are set by the cell above.
from urllib.parse import quote as _quote

encoded_id = _quote(EXT_REF_ID, safe="")
r = requests.get(
    urljoin(BASE_URL, f"ExternalReferences/{encoded_id}/{EXT_REF_PARENT_UUID}"),
    headers=HEADERS,
    timeout=30,
)
if not r.ok:
    print(f"Request failed: HTTP {r.status_code}")
    print(r.text)
else:
    print(json.dumps(r.json(), indent=2))

In [ ]:
# Delete an External Reference using the composite key (id, parent_uuid).
# EXT_REF_ID and EXT_REF_PARENT_UUID are set by earlier cells; set them manually if needed:
# EXT_REF_ID          = "jira:ABC-123"
# EXT_REF_PARENT_UUID = "<paste-feature-uuid>"

if USE_SANDBOX:
    print(
        "Sandbox mode is read-only: DELETE /ExternalReferences/{id}/{parent_uuid} is not allowed. "
        "Set USE_SANDBOX = False and configure OAuth credentials in apidata.py to delete external references."
    )
else:
    from urllib.parse import quote
    encoded_id = quote(EXT_REF_ID, safe="")
    delete_url = urljoin(BASE_URL, f"ExternalReferences/{encoded_id}/{EXT_REF_PARENT_UUID}")
    r = requests.delete(delete_url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    print(f"Deleted External Reference (id='{EXT_REF_ID}', parent_uuid='{EXT_REF_PARENT_UUID}') — HTTP {r.status_code}")

## 5. Task Assignments (read-only)

Task Assignments expose the project tasks, user stories, defects, and requirements linked to a Feature.
This relationship is managed in SAP Cloud ALM's task management area — it is **read-only** via the Features API.

Task types returned in the `type` field:

| Type | Meaning |
|------|----------|
| `CALMTASK` | Project task |
| `CALMUS` | User story |
| `CALMDEF` | Defect |
| `CALMREQU` | Requirement |

In [ ]:
from urllib.parse import urlencode, quote

PAGE_SIZE = 50
MAX_PAGES = 5

base_params = {
    "$top": str(PAGE_SIZE),
    "$select": "uuid,parent_uuid,title,type",
    "$orderby": "title asc",
    "$count": "true",
}

def _odata_get(url: str, params: dict):
    query = urlencode(params, quote_via=quote)
    return requests.get(f"{url}?{query}", headers=HEADERS, timeout=30)

all_tasks   = []
page        = 0
next_url    = urljoin(BASE_URL, f"Features/{FEATURE_UUID}/toTaskAssignments")
use_orderby = True

while next_url and page < MAX_PAGES:
    if page == 0:
        request_params = base_params if use_orderby else {k: v for k, v in base_params.items() if k != "$orderby"}
        r = _odata_get(next_url, request_params)
        if r.status_code == 400 and use_orderby:
            print("$orderby rejected; retrying first page without $orderby")
            use_orderby    = False
            request_params = {k: v for k, v in base_params.items() if k != "$orderby"}
            r              = _odata_get(next_url, request_params)
    else:
        r = requests.get(next_url, headers=HEADERS, timeout=30)

    r.raise_for_status()
    body  = r.json()
    batch = body.get("value", [])
    all_tasks.extend(batch)

    if page == 0:
        print(f"Total task assignments: {body.get('@odata.count', 'n/a')}")
    print(f"  Page {page + 1}: {len(batch)} item(s) retrieved")

    next_url = body.get("@odata.nextLink")
    page    += 1

print(f"\nFetched {len(all_tasks)} task assignment(s) across {page} page(s)")
print(json.dumps(all_tasks[:3], indent=2))

## 6. Transports and Transport References (read-only)

Transports and Transport References show the change transport items linked to a Feature.
These are populated by SAP Cloud ALM's transport management workflows and are **read-only** via the Features API.

In [ ]:
for nav in ("toTransports", "toTransportReferences"):
    r = requests.get(
        urljoin(BASE_URL, f"Features/{FEATURE_UUID}/{nav}"),
        headers=HEADERS,
        timeout=30,
    )
    r.raise_for_status()
    items = r.json().get("value", [])
    print(f"{nav}: {len(items)} item(s)")
    print(json.dumps(items[:2], indent=2))
    print()